In [1]:
%useLatestDescriptors

In [40]:
%use datetime
%use dataframe
%use kandy
%use ktor-client
%use coroutines

In [3]:
import kotlinx.datetime.format.byUnicodePattern
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import kotlin.time.Clock
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"

val configData = DataRow.readJson(path=serviceKeyFilePath)

In [55]:
import com.unchil.oceanwaterinfo.Config.Companion.configData

val now = Clock.System.now()

@OptIn(FormatStringsInDatetimeFormats::class)
val currentTime = now
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH:mm:ss")})

@OptIn(FormatStringsInDatetimeFormats::class)
val previous24Hour = now
    .minus(1, DateTimeUnit.HOUR)
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd")})

print("Current time : ${currentTime}, Previous time : ${previous24Hour}")


val url = "${configData.MOF_API?.endPoint}/${configData.MOF_API?.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(previous24Hour, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(currentTime, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"


Current time : 2026-07-20 14:39:21, Previous time : 2026-07-20

In [5]:
@file:DependsOn("org.json:json:20250107")

In [6]:
import io.ktor.client.HttpClient
import io.ktor.client.engine.ProxyBuilder.http
import io.ktor.client.engine.cio.CIO
import io.ktor.client.plugins.HttpTimeout
import io.ktor.client.plugins.contentnegotiation.ContentNegotiation
import io.ktor.client.request.get
import io.ktor.client.statement.bodyAsText
import io.ktor.serialization.kotlinx.json.json
import org.json.XML
import kotlin.coroutines.suspendCoroutine

val maxPage = 500

In [70]:
@Serializable
@SerialName("item")
data class OceanWaterQuality (
    val num: Int, // 순번
    val rtmWqWtchStaCd: Double, // 실시간수질관측정점코드
    val rtmWqWtchDtlDt: String, // 실시간수질관측상세일시
    val rtmWtchWtem: Double, // 실시간관측수온
    val rtmWqCndctv: Double, // 실시간수질전기전도도
    val ph: Double, // 수소이온농도
    val rtmWqDoxn: Double, // 실시간수질용존산소량
    val rtmWqTu: Double, // 실시간수질탁도
    val rtmWqBgalgsQy: Double?, // 실시간수질남조류량
    val rtmWqChpla: Double, // 실시간수질클로로필
    val rtmWqSlnty: Double // 실시간수질염분
)



In [83]:
fun loadData(path:String):DataFrame<OceanWaterQuality> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<OceanWaterQuality>>()

    do{
        val pagePath = "$path&pageNo=$requestPage"
        try {
            val response = http.get(pagePath)

            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<OceanWaterQuality>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    print(e.localizedMessage)
                    break
                }

            } else {
                println("${response.status.description}")
            }

        } catch(e:Exception ){
            requestPage += 1
            println(e.localizedMessage)

        }

    } while (requestPage < maxPage )

    return rows.concat()

}


In [86]:
val df = loadData(url)
df.head(5)

Column not found: 'items'

rtmWqDoxn,rtmWqChpla,rtmWqBgalgsQy,rtmWqWtchStaCd,num,rtmWqTu,ph,rtmWqSlnty,rtmWqCndctv,rtmWqWtchDtlDt,rtmWtchWtem
4.057000,2.041000,,NEP2001,1,14,7.140000,1.276000,2.478000,2026-07-20 00:00:00.0,27.230000
2.892000,1.457000,,NEP3001,2,9,7.420000,13.396000,22.264999,2026-07-20 00:00:00.0,25.920000
5.430000,10.680000,,SEA6001,3,43,7.810000,31.197001,47.959000,2026-07-20 00:00:00.0,26.950001
6.610000,21.660000,,NEP2002,4,8,7.980000,17.900000,29.200001,2026-07-20 00:00:00.0,25.320000
5.500000,6.890000,,SEA5003,5,12,7.620000,32.582001,48.202000,2026-07-20 00:00:00.0,23.370001
